# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faculatini/flyrank_ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data contract

**Unit of analysis**

One row represents one content page for one client on one reporting date.

**Table(s)**

This notebook uses the `fact_content_daily_performance` table from the FlyRank Internship Warehouse.

**Time window**

This analysis uses data from **March 2026**, a mid-panel month, to avoid using the final month during development.

**Target / proxy**

The project aims to rank pages by editorial review priority. Since the warehouse does not contain the outcome of future editorial actions, a proxy label will be used during development instead of a true future outcome.

**Deliberately excluded**

Any field derived from the target, any future information, and any identifier columns (`client_hash_id`, `content_hash_id`) are excluded from the model.

**Output**

The output is a ranked list of pages that deserve editorial review, helping editors prioritize where to spend their limited review time.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [3]:
DATASET = "hf://datasets/FlyRank/internship-warehouse"

In [4]:
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
'{DATASET}/fact_content_daily_performance/**/*.parquet'
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [5]:
march = f"""
SELECT *
FROM read_parquet(
'{DATASET}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
"""
df = con.sql(march).df()
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [6]:
print(df.head())

  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e   
4  2026-03-01  client_73cda7b4e4f265ea  content_a3ea9792f793ec72   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True           False                True                <NA>   
1            True           False                True                <NA>   
2            True           False                True                <NA>   
3            True           False                True                <NA>   
4            True           False                True                <NA>   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               20           0                67  ...     

In [9]:
query1 = con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS n
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
""")

query1.df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


In [10]:
query2 = con.sql("""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""")

query2.df()

,rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [11]:
query3 = con.sql("""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE
    month = '2026-03'
    AND gsc_data_available IS TRUE
""")

query3.df()

,available_rows
0,3611061


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.